# Beautiful Soup: turning HTML into structured records

A bookstore has saved catalog pages, but a spreadsheet is needed to compare prices and availability. Work from a repeated book card to a validated record, then across pages to an export. Decide what identifies a book, which fields are optional, and how you will notice a selector that has stopped matching.

Requests retrieves a page. Beautiful Soup parses its HTML into a tree so we can find headings, follow relationships between elements, and extract fields into records.

We will work with an **invented bookstore page**, then extract two saved pages into JSON and CSV. The examples are offline and make no network requests. Run cells from top to bottom. Beautiful Soup does not execute JavaScript or act as a browser.

## 1. Install and import

If needed, run `%pip install beautifulsoup4` in a separate cell. Install `beautifulsoup4`, then import from `bs4`. We explicitly choose Python's built-in `html.parser`, so no additional parser package is needed.

Different parsers can repair malformed HTML differently. Naming the parser makes the intended behavior clearer and more reproducible.

In [ ]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from urllib.parse import urljoin, urlsplit, urldefrag
import csv
import json
import bs4
from bs4 import BeautifulSoup

print("Beautiful Soup:", bs4.__version__)
BASE_URL = "https://books.example/catalog/page1.html"

## 2. Read the page structure

HTML contains tags, attributes, and text. A repeated `article` represents one book. Its `data-id` identifies the record, its heading contains a link, and its price and stock are separate elements.

The sample deliberately includes nested text, an HTML entity, an absent price, duplicate navigation links, and a small table. These are common extraction cases.

In [ ]:
HTML = """<!doctype html>
<html lang="en"><head><meta charset="utf-8"><title>Practice Books</title>
<style>.promo {color: purple}</style></head><body>
<header><h1>Practice Books</h1><nav><a href="/">Home</a><a href="#books">Books</a></nav></header>
<main id="books">
<p class="intro">Find <strong>useful</strong> books &amp; more.</p>
<article class="book featured" data-id="b1">
<h2><a href="../book/python-basics.html">Python <em>Basics</em></a></h2>
<p class="price" data-currency="USD">$19.50</p><p class="stock">In stock</p>
<p class="description">A practical introduction.</p></article>
<article class="book" data-id="b2">
<h2><a href="/book/data-tables.html">Data &amp; Tables</a></h2>
<p class="price" data-currency="USD">$24.00</p><p class="stock">Out of stock</p></article>
<article class="book" data-id="b3">
<h2><a href="/book/visual-stories.html">Visual Stories</a></h2>
<p class="stock">In stock</p></article>
<table id="hours"><thead><tr><th>Day</th><th>Hours</th></tr></thead>
<tbody><tr><td>Monday</td><td>09:00–17:00</td></tr><tr><td>Saturday</td><td>10:00–14:00</td></tr></tbody></table>
</main>
<footer><a href="/book/data-tables.html#details">Featured book</a>
<a href="mailto:hello@books.example">Email</a><a href="https://other.example/about">Partner</a>
<a rel="next" href="page2.html">Next page</a></footer>
<script type="application/ld+json">{"@context":"https://schema.org","@type":"BookStore","name":"Practice Books"}</script>
</body></html>"""
soup = BeautifulSoup(HTML, "html.parser")
print("Page title:", soup.title.get_text())
print("Book cards:", len(soup.select("article.book")))

## 3. Locate a book before collecting its fields

Before running the selector, predict whether it should return one element or several. Inspect the result inside one book card. If a page contains repeated headings, a globally correct-looking title may still belong to the wrong book.

`find` and `select_one` return one matching element, or `None` if none exists. `find_all` and `select` return collections, which may be empty.

Use `class_` with `find` because `class` is a Python keyword. Search within a container when you want fields from the same record.

In [ ]:
first_book = soup.find("article", class_="book")
all_books = soup.find_all("article", class_="book")
print("First ID:", first_book.get("data-id"))
print("All IDs:", [book.get("data-id") for book in all_books])
print("Missing element:", soup.find(id="does-not-exist"))

## 4. Use CSS selectors

| Selector | Meaning |
| --- | --- |
| `article.book` | An article with class book |
| `#books` | The element with ID books |
| `article.book h2 a` | Links inside book headings |
| `article.book > p.price` | Price paragraphs directly inside a book |
| `[data-id="b2"]` | An element with this attribute value |
| `article.featured` | Featured articles |

Prefer semantic classes and attributes to long position-based selectors that break when an unrelated element is inserted.

In [ ]:
heading_links = soup.select("article.book h2 a")
print("Titles:", [link.get_text(" ", strip=True) for link in heading_links])
print("Prices:", [node.get_text(strip=True) for node in soup.select("article.book > p.price")])
print("Featured:", soup.select_one("article.featured h2").get_text(" ", strip=True))
print("Second book:", soup.select_one('[data-id="b2"] h2').get_text(" ", strip=True))

## 5. Extract readable text

`.string` is useful for simple text nodes, but can be `None` when a tag contains multiple children. `get_text(" ", strip=True)` joins nested text with spaces and trims each piece. Splitting and rejoining also normalizes repeated whitespace for these plain-text fields.

HTML entities are decoded by the parser. Whitespace normalization is a choice: preserve it when extracting code, poetry, or other formatting-sensitive content.

In [ ]:
def readable_text(node):
    return " ".join(node.get_text(" ", strip=True).split())

intro = soup.select_one(".intro")
print("Direct string:", intro.string)
print("Readable text:", readable_text(intro))
print("Nested title:", readable_text(first_book.h2))
print("Decoded entity:", readable_text(soup.select_one('[data-id="b2"] h2')))

## 6. Read attributes and navigate relationships

`tag["href"]` requires an attribute and raises an error if it is absent. `tag.get("href")` can return `None`. HTML class attributes are represented as lists.

Parents, children, and siblings describe the tree. `next_sibling` may be a whitespace text node; `find_next_sibling` searches for a matching element.

In [ ]:
first_link = first_book.select_one("h2 a")
print("Link target:", first_link["href"])
print("Optional title attribute:", first_link.get("title"))
print("Classes:", first_book.get("class"))
print("Containing book:", first_link.find_parent("article")["data-id"])
print("Next paragraph:", readable_text(first_book.h2.find_next_sibling("p")))
print("Direct paragraph count:", len(first_book.find_all("p", recursive=False)))

## 7. Resolve relative links

`urljoin` resolves a relative link using the source page URL. A root-relative path starts from the site's root; `../` moves up a path level. It does not fetch anything.

For this exercise, retain same-origin HTTP(S) links, remove fragments, and deduplicate while preserving order. Removing a fragment does not remove query parameters. A different extraction task may need a different link policy.

In [ ]:
def local_http_link(href, page_url):
    if not href or not href.strip():
        return None
    absolute = urldefrag(urljoin(page_url, href.strip()))[0]
    parsed = urlsplit(absolute)
    origin = urlsplit(page_url)
    if parsed.scheme not in ("http", "https") or (parsed.scheme, parsed.netloc) != (origin.scheme, origin.netloc):
        return None
    return absolute

resolved_links = list(dict.fromkeys(
    target for link in soup.select("a[href]")
    if (target := local_http_link(link.get("href"), BASE_URL)) is not None
))
print("Resolved first book:", local_http_link(first_link["href"], BASE_URL))
print("Unique same-origin links:", resolved_links)

## 8. Keep an absent price different from a free book

Predict how an absent price should appear in the exported data. Compare missing and malformed values: one means no observation, the other means the parser cannot interpret an observation. Neither justifies substituting zero.

The third book has no listed price. Missing is represented by `None`, not zero. A price element with unsupported currency or malformed text is a different problem and raises an error.

`Decimal` preserves decimal currency values. This parser intentionally supports only the sample's USD format with a dollar sign; other locales and formats need explicit rules.

In [ ]:
def parse_price(node):
    if node is None:
        return None
    text = readable_text(node)
    if node.get("data-currency") != "USD" or not text.startswith("$"):
        raise ValueError("Expected a USD price with a dollar sign")
    try:
        value = Decimal(text[1:])
    except InvalidOperation as error:
        raise ValueError(f"Invalid price: {text}") from error
    if not value.is_finite() or value < 0:
        raise ValueError("Price must be finite and nonnegative")
    return value

print("First price:", parse_price(first_book.select_one(".price")))
print("Missing price:", parse_price(soup.select_one('[data-id="b3"] .price')))

## 9. Prevent titles and prices from drifting apart

Imagine that the middle book has no price. Separate lists of titles and prices would have different lengths. Trace each extracted field back to its parent card and verify that the resulting record preserves that association.

Keep a title, price, and stock status within the same card. Extracting separate global lists and zipping them would misalign records when a field is missing.

Required fields raise clear errors. Optional descriptions remain `None`. Match stock labels exactly: the text “Out of stock” also contains the word “stock”.

In [ ]:
def extract_book(card, page_url):
    book_id = card.get("data-id")
    title_link = card.select_one("h2 a")
    stock_node = card.select_one(".stock")
    if not book_id or title_link is None or stock_node is None:
        raise ValueError("Book is missing an ID, title link, or stock status")
    title = readable_text(title_link)
    url = local_http_link(title_link.get("href"), page_url)
    if not title or url is None:
        raise ValueError(f"Invalid title or book link for {book_id}")
    stock = readable_text(stock_node)
    stock_values = {"In stock": True, "Out of stock": False}
    if stock not in stock_values:
        raise ValueError(f"Unknown stock label: {stock}")
    price = parse_price(card.select_one(".price"))
    description_node = card.select_one(".description")
    return {"id": book_id, "title": title, "price_usd": str(price) if price is not None else None,
            "in_stock": stock_values[stock], "url": url,
            "description": readable_text(description_node) if description_node is not None else None,
            "source_url": page_url}

book_records = [extract_book(card, BASE_URL) for card in soup.select("article.book")]
print(json.dumps(book_records, indent=2, ensure_ascii=False))

## 10. Parse a simple table

Use header labels as dictionary keys and check that each row has the expected number of cells. This example has a rectangular table without merged cells.

Tables with `rowspan`, `colspan`, multiple header rows, or nested tables need extra handling; simply collecting cells is not enough for those structures.

In [ ]:
hours_table = soup.select_one("table#hours")
headers = [readable_text(node) for node in hours_table.select("thead th")]
hours_records = []
for row in hours_table.select("tbody > tr"):
    values = [readable_text(cell) for cell in row.find_all("td", recursive=False)]
    if len(values) != len(headers):
        raise ValueError("Unexpected table width")
    hours_records.append(dict(zip(headers, values)))
print(hours_records)

## 11. Read embedded JSON

Some pages include structured metadata in a script tag with type `application/ld+json`. Extract its contents and parse them with `json.loads`; do not execute the script.

Real JSON-LD can contain lists, nested objects, or graphs. This page contains one small object, and its metadata describes the store rather than every book.

In [ ]:
metadata_node = soup.find("script", attrs={"type": "application/ld+json"})
store_metadata = json.loads(metadata_node.string)
print("Metadata type:", store_metadata["@type"])
print("Store name:", store_metadata["name"])

## 12. Modify a copy of the tree

`decompose` removes a tag and its contents; `unwrap` removes a wrapper while preserving its children. These methods mutate the tree, so parse a separate copy for cleanup.

We remove navigation and scripts, unwrap an emphasis tag, and update the heading. This demonstrates editing, not a general-purpose HTML sanitizer.

In [ ]:
clean_soup = BeautifulSoup(HTML, "html.parser")
for unwanted in clean_soup.select("script, style, nav, footer"):
    unwanted.decompose()
clean_soup.select_one("article.book em").unwrap()
clean_soup.h1.string = "Book catalog extract"
print("Updated heading:", readable_text(clean_soup.h1))
print("Original heading:", readable_text(soup.h1))
print("Remaining script count:", len(clean_soup.select("script")))

## 13. Treat parser recovery as something to inspect

Parsers can build a tree from imperfect HTML, but the resulting hierarchy may differ across parsers. Inspect the output instead of assuming it matches a browser's document tree.

This deliberately unclosed paragraph still has recoverable text. For reproducible extraction, keep the parser choice explicit and retain source samples that expose edge cases.

In [ ]:
imperfect_html = "<div><p>A note with <b>bold text</b>"
recovered = BeautifulSoup(imperfect_html, "html.parser")
print(recovered.prettify())
print("Recovered text:", readable_text(recovered))

## 14. Know when the catalog is complete

Before following the next-page link, define a stopping rule. After traversal, compare page visits, unique book IDs and record counts. An empty result can mean no books or a broken selector; the saved HTML provides evidence to distinguish them.

A next-page link describes navigation; Beautiful Soup does not follow it automatically. Here a dictionary supplies two saved HTML pages, so we can practice pagination without a live site.

Keep the source URL on every record, reject repeated page visits and duplicate record IDs, and bound the number of pages. Unexpected empty catalogs raise an error rather than silently looking like success.

In [ ]:
PAGE_TWO = """<html><body><main>
<article class="book" data-id="b4"><h2><a href="/book/practical-sql.html">Practical SQL</a></h2>
<p class="price" data-currency="USD">$29.00</p><p class="stock">In stock</p></article>
</main></body></html>"""
PAGE_TWO_URL = urljoin(BASE_URL, "page2.html")
saved_pages = {BASE_URL: HTML, PAGE_TWO_URL: PAGE_TWO}

def extract_catalog(pages, start_url, max_pages=10):
    records, visited, seen_ids = [], set(), set()
    page_url = start_url
    for _ in range(max_pages):
        if page_url in visited:
            raise ValueError("Repeated pagination URL")
        if page_url not in pages:
            raise ValueError(f"No saved HTML for {page_url}")
        visited.add(page_url)
        page = BeautifulSoup(pages[page_url], "html.parser")
        cards = page.select("article.book")
        if not cards:
            raise ValueError("Expected book cards on this catalog page")
        for card in cards:
            record = extract_book(card, page_url)
            if record["id"] in seen_ids:
                raise ValueError(f"Duplicate book ID: {record['id']}")
            seen_ids.add(record["id"])
            records.append(record)
        next_link = page.select_one('a[rel~="next"]')
        if next_link is None:
            return records
        next_url = local_http_link(next_link.get("href"), page_url)
        if next_url is None:
            raise ValueError("Invalid next-page link")
        page_url = next_url
    raise RuntimeError("Pagination exceeded max_pages")

catalog_records = extract_catalog(saved_pages, BASE_URL)
print("Extracted books:", len(catalog_records))
print("Missing prices:", sum(record["price_usd"] is None for record in catalog_records))
print("In stock:", sum(record["in_stock"] for record in catalog_records))

## 15. Export records and retain source HTML

JSON preserves the missing price as `null`; CSV stores it as an empty field. Prices are decimal strings, so another program can load them into Decimal without binary floating-point rounding.

Save the source pages alongside the extract to make selector problems reproducible. Running this cell creates `outputs` and replaces these named files.

In [ ]:
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)
json_path = output_dir / "beautifulsoup_books.json"
csv_path = output_dir / "beautifulsoup_books.csv"
json_path.write_text(json.dumps(catalog_records, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(catalog_records[0]))
    writer.writeheader()
    writer.writerows(catalog_records)
for number, source in enumerate([HTML, PAGE_TWO], start=1):
    (output_dir / f"beautifulsoup_source_{number}.html").write_text(source, encoding="utf-8")
(output_dir / "beautifulsoup_cleaned.html").write_text(str(clean_soup), encoding="utf-8")
print("Saved:", json_path)
print("Saved:", csv_path)
print("Saved two source pages and the edited HTML copy.")

## 16. Connect to Requests when working with a live page

The previous Requests notebook handles HTTP. The boundary is straightforward: retrieve the response, check its status, then parse the HTML. This illustrative snippet is not executed by this notebook:

```python
response = session.get(page_url, timeout=(3, 10))
response.raise_for_status()
page = BeautifulSoup(response.content, "html.parser")
records = [extract_book(card, response.url) for card in page.select("article.book")]
```

Using the final response URL resolves relative links after redirects. Real pages may specify a `<base href>` that also needs to be considered; our sample does not. Selectors must match the actual site's markup. If content exists only after JavaScript runs, it will not appear just because Beautiful Soup parses the initial response.

## 17. Practice and troubleshooting

1. Add a book with no description and confirm its other fields stay aligned.
2. Change a stock label and verify that the extractor reports it instead of guessing.
3. Point page two back to page one and check the cycle detection.
4. Add a duplicate ID to page two and inspect the validation error.

| Symptom | What to check |
| --- | --- |
| `NoneType` has no `get_text` | The selector found nothing; inspect the source and distinguish required from optional fields |
| Words run together | Supply a separator to `get_text` |
| Price and title lists have different lengths | Extract fields within each record container |
| Link points to the wrong page | Resolve it against the correct source URL, considering any base element |
| Browser shows more records than extracted HTML | Check whether JavaScript or another request supplies them |
| CSV loses missing-value meaning | Define the schema and missing-value convention explicitly |
| Another machine produces a different tree | Match parser and package versions, especially for invalid HTML |

See the official [Beautiful Soup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) for searching, CSS selectors, tree editing, encodings, and parser differences.

## Explain what the evidence supports

Choose a card whose price is absent and trace its title, ID and price from HTML into the export. Explain how you would detect a page redesign without silently publishing an empty catalog.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: extract a catalog after the HTML layout changes

A saved two-page catalog contains inconsistent class names, a repeated product, missing optional prices, unsupported currency and a card without a title. Build record-level extraction and a rejection audit, then compare parsing once with reparsing for each field. All HTML is authored locally; no website access or browser automation is needed.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through why parallel field lists corrupt a table

If three cards have titles but only two have prices, zipping global title and price lists silently drops one card and may attach the wrong price to another. Extract within each card, carry its stable SKU, and distinguish missing price from malformed price. Missing does not mean free.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from bs4 import BeautifulSoup
from decimal import Decimal,InvalidOperation
p2_html='''<main>
<article class="book" data-sku="b1"><h2>Data Basics</h2><span class="price">$12.50</span></article>
<article class="book" data-sku="b2"><h2>SQL Practice</h2></article>
<article class="book" data-sku="b3"><h2>Model Design</h2><span class="price">$20.00</span></article>
</main>'''
p2_next='''<main>
<article class="product" data-sku="b3"><h2>Model Design</h2><span class="price">$20.00</span></article>
<article class="product" data-sku="b4"><h2>European Edition</h2><span class="price">EUR 10</span></article>
<article class="product" data-sku="b5"><span class="price">$15.00</span></article>
</main>'''
p2_soup=BeautifulSoup(p2_html,'html.parser')
p2_wrong=list(zip([t.get_text(strip=True) for t in p2_soup.select('h2')],[t.get_text(strip=True) for t in p2_soup.select('.price')]))
assert len(p2_wrong)==2 and p2_wrong[1]==('SQL Practice','$20.00')
print('Misaligned global extraction:',p2_wrong)

### Work through an auditable extractor

Use the semantic `data-sku` attribute rather than a presentation class that changed from book to product. Reject unsupported currencies and missing required titles, but retain the optional missing price. A repeated SKU is a retry only if its extracted content agrees; conflicting content needs a version policy.

For performance, parse once and reuse the tree for several selectors. Repeated parsing is extra CPU work, while retaining every tree forever wastes memory. The example releases each page after extracting ordinary records.

In [ ]:
def p2_extract(card):
    p2_title=card.select_one('h2')
    if p2_title is None or not p2_title.get_text(strip=True):raise ValueError('missing title')
    p2_price=card.select_one('.price');p2_value=None
    if p2_price is not None:
        p2_text=p2_price.get_text(strip=True)
        if not p2_text.startswith('$'):raise ValueError('unsupported currency')
        p2_value=Decimal(p2_text[1:])
        if not p2_value.is_finite() or p2_value<0:raise ValueError('invalid price')
    return {'sku':card['data-sku'],'title':p2_title.get_text(' ',strip=True),'price_usd':None if p2_value is None else str(p2_value)}
p2_records={};p2_rejects=[];p2_duplicates=0
for p2_page,p2_body in enumerate([p2_html,p2_next],1):
    p2_tree=BeautifulSoup(p2_body,'html.parser')
    for p2_card in p2_tree.select('article[data-sku]'):
        try:p2_record=p2_extract(p2_card)
        except (ValueError,InvalidOperation) as p2_error:p2_rejects.append({'page':p2_page,'sku':p2_card.get('data-sku'),'reason':str(p2_error)});continue
        if p2_record['sku'] in p2_records:
            assert p2_records[p2_record['sku']]==p2_record,'Conflicting SKU versions'
            p2_duplicates+=1
        else:p2_records[p2_record['sku']]=p2_record
assert len(p2_records)==3 and len(p2_rejects)==2 and p2_duplicates==1
assert p2_records['b2']['price_usd'] is None
p2_timings=[]
for p2_reparse in [True,False]:
    p2_t=time.perf_counter()
    for _ in range(100):
        p2_tree=BeautifulSoup(p2_html,'html.parser')
        for p2_selector in ['h2','.price','article[data-sku]']:
            p2_result=(BeautifulSoup(p2_html,'html.parser') if p2_reparse else p2_tree).select(p2_selector)
    p2_timings.append({'reparse_each_selector':p2_reparse,'seconds':time.perf_counter()-p2_t})
pd.DataFrame(p2_records.values()).to_csv(p2_out/'bs4_catalog.csv',index=False)
pd.DataFrame(p2_rejects).to_csv(p2_out/'bs4_rejections.csv',index=False)
display(pd.DataFrame(p2_records.values()));display(pd.DataFrame(p2_timings))

### Explain the result and the remaining fragility

The missing-price item survives with the correct title; the repeated item does not double the count. Semantic attributes are often more stable than CSS classes, but they can also change. Preserve source samples and compare expected card counts rather than assuming a selector returning zero is a valid empty catalog.

**Try:** change b3's second price. **Worked reasoning:** the conflict assertion should stop publication. Keeping an arbitrary last row hides a disagreement. A real feed needs source timestamps or a documented priority rule. The timing exercise isolates parser reuse on tiny saved pages; network and JavaScript rendering costs are outside it.